# Signal Hunters R2 - run 3: ETI injection + pseudo-labels on confident PKS test cadences (domain adaptation)
GPU T4 x2, Internet on, input: signal-hunters-r2. Self-contained.

In [ ]:
%%writefile /kaggle/working/cnn_r2.py
"""Signal Hunters R2 - CNN on the 'global-time canvas'.
Each cadence -> one 128x128 image: the 6 panels placed on ONE time axis including the slew gaps
(panel p row r at t = p*(16+gap)+r), so a true ETI drift is a single straight line through the 3 ON panels.
Input channels: z (bandpass-flattened, noise-normalised), zr (z minus per-row median: kills broadband rows),
on/off indicator (+1 ON rows, -1 OFF rows, 0 gaps), valid mask.
Domain shift GBT->PKS is attacked with augmentation: frequency zoom (PKS drifts look ~0.64x as steep),
SNR loss, dropped rows, broadband glitch rows, per-panel gain, frequency shift/flip.
Usage: python cnn_r2.py DATA_DIR EPOCHS SEEDS [arch=res|timm_resnet18|timm_effb0] [PL]
Writes oof_<tag>.npy (18000x9) and test_<tag>.npy (10000x9) to ./out/"""
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn, sys, os, time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
LABELS = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass', 'radar_chirp',
          'rfi_intermittent', 'rfi_persistent', 'noise']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
P_ETI = float(os.environ.get('P_ETI', 0)); P_TRAP = float(os.environ.get('P_TRAP', 0))

def normalise(X):
    x = X.astype(np.float32); m = (X > 0); x[~m] = np.nan
    x = x - np.nanmedian(x, axis=2, keepdims=True)
    s = 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x, axis=(2, 3), keepdims=True)), axis=(2, 3), keepdims=True) + 1e-3
    z = np.nan_to_num(np.clip(x / s, -10, 60))
    return z.astype(np.float16), m

def load(D, split):
    X = np.load(f'{D}/{split}_cadences.npy', mmap_mode='r'); meta = pd.read_csv(f'{D}/{split}_meta.csv')
    Z = np.empty(X.shape, np.float16); M = np.empty(X.shape, bool)
    for s in range(0, len(X), 2000):
        Z[s:s + 2000], M[s:s + 2000] = normalise(np.asarray(X[s:s + 2000]))
    return Z, M, meta.slew_gap_rows.values.astype(np.int64), meta

# ---------------- GPU-side augmentation + canvas -----------------
def augment(z, m, train):
    """z,m: (B,6,16,128) float on device"""
    B = z.shape[0]
    if not train: return z, m
    # SNR loss + extra noise (PKS noisier): z' = a*z + sqrt(1-a^2)*N
    a = torch.empty(B, 1, 1, 1, device=z.device).uniform_(0.55, 1.0)
    z = a * z + torch.sqrt(1 - a ** 2) * torch.randn_like(z)
    # per-panel gain on the signal part
    z = z * torch.empty(B, 6, 1, 1, device=z.device).uniform_(0.8, 1.2)
    # frequency zoom (compress/expand channel axis), p=0.7, s in [0.55,1.1]
    sel = torch.rand(B, device=z.device) < 0.7
    if sel.any():
        s = torch.ones(B, device=z.device); s[sel] = torch.empty(int(sel.sum()), device=z.device).uniform_(0.55, 1.1)
        xs = torch.linspace(-1, 1, 128, device=z.device)
        gx = xs[None, :] / s[:, None]                                   # sample positions in input
        grid = torch.stack([gx[:, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=z.device)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, mode='bilinear', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        mm = Fn.grid_sample(m.reshape(B, 1, 96, 128).float(), grid, mode='nearest', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        outside = (gx.abs() > 1)[:, None, None, :].expand(B, 6, 16, 128)
        # bilinear interpolation shrinks noise std -> renormalise per panel, fill outside with fresh noise
        zz = torch.where(outside, torch.randn_like(zz), zz)
        mm = torch.where(outside, torch.ones_like(mm), mm)
        sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        z = torch.where(sel[:, None, None, None], zz, z); m = torch.where(sel[:, None, None, None], mm, m.float())
    m = m.float()
    # frequency shift (roll) and flip
    sh = int(np.random.randint(-30, 31)); z = torch.roll(z, sh, -1); m = torch.roll(m, sh, -1)
    fl = torch.rand(B, device=z.device) < 0.5
    z = torch.where(fl[:, None, None, None], z.flip(-1), z); m = torch.where(fl[:, None, None, None], m.flip(-1), m)
    # dropped rows
    drop = (torch.rand(B, 6, 16, 1, device=z.device) < torch.empty(B, 1, 1, 1, device=z.device).uniform_(0, 0.04)).float()
    m = m * (1 - drop); z = z * (1 - drop)
    # broadband glitch rows
    gl = (torch.rand(B, 6, 16, 1, device=z.device) < 0.015).float() * torch.empty(B, 6, 16, 1, device=z.device).uniform_(1, 8)
    z = z + gl
    return z, m

def canvas(z, m, gap):
    """(B,6,16,128) -> (B,4,128,128) global-time canvas"""
    B = z.shape[0]; H = 128
    zr = z - z.median(-1, keepdim=True).values
    C = torch.zeros(B, 4, H, 128, device=z.device)
    p = torch.arange(6, device=z.device); r = torch.arange(16, device=z.device)
    t = p[None, :, None] * (16 + gap[:, None, None]) + r[None, None, :]          # (B,6,16)
    t = t + ((H - (6 * 16 + 5 * gap)) // 2)[:, None, None]                       # centre vertically
    bi = torch.arange(B, device=z.device)[:, None, None].expand(B, 6, 16)
    sign = torch.tensor([1., -1, 1, -1, 1, -1], device=z.device)[None, :, None].expand(B, 6, 16)
    C[bi, 0, t] = z * m; C[bi, 1, t] = zr * m
    C[bi, 2, t] = sign[..., None].expand(B, 6, 16, 128); C[bi, 3, t] = m
    return C


def inject(z, m, gap, yb, p_eti, p_trap):
    """Physics-based synthetic signals (labels follow the priority order):
    - ETI: a narrow drifting line on ONE straight track in global time, ON panels only -> label eti_candidate
    - trap: ON-only lines that do NOT connect (random offsets) or do not drift -> rfi_intermittent
      (only injected into noise / rfi_persistent cadences, whose priority is lower)"""
    B = z.shape[0]; d_ = z.device
    t = (torch.arange(6, device=d_)[None, :, None] * (16 + gap[:, None, None]) + torch.arange(16, device=d_)[None, None, :]).float()
    tc = t[:, [0, 2, 4]].mean((1, 2))[:, None, None]
    ch = torch.arange(128, device=d_).float()[None, None, None, :]
    on = torch.tensor([1., 0, 1, 0, 1, 0], device=d_)[None, :, None, None]
    r = torch.rand(B, device=d_)
    e = (r < p_eti) & (yb != 0)
    tr_ = (r >= p_eti) & (r < p_eti + p_trap) & ((yb == 7) | (yb == 8))
    if e.any() or tr_.any():
        drift = torch.empty(B, device=d_).uniform_(0.15, 1.3) * (torch.randint(0, 2, (B,), device=d_) * 2 - 1)
        c0 = torch.empty(B, device=d_).uniform_(20, 108)
        amp = torch.empty(B, device=d_).uniform_(0.6, 3.0); wid = torch.empty(B, device=d_).uniform_(0.5, 1.2)
        centre = c0[:, None, None] + drift[:, None, None] * (t - tc)                      # (B,6,16) connected track
        kind = torch.randint(0, 2, (B,), device=d_)                                       # trap type
        off = torch.empty(B, 6, 1, device=d_).uniform_(-40, 40)                           # disconnect offsets
        tr_centre = torch.where(kind[:, None, None] == 0, c0[:, None, None] + off + drift[:, None, None] * (t - t[:, :, :1]),
                                c0[:, None, None] + 0 * t)                                # non-connecting / non-drifting
        cen = torch.where(e[:, None, None], centre, tr_centre)
        line = amp[:, None, None, None] * torch.exp(-0.5 * ((ch - cen[..., None]) / wid[:, None, None, None]) ** 2) * on
        sel = (e | tr_)[:, None, None, None].float()
        z = z + sel * line * m
        yb = torch.where(e, torch.zeros_like(yb), torch.where(tr_, torch.full_like(yb, 6), yb))
    return z, yb

# ---------------- model -----------------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o)) if (i != o or st != 1) else nn.Identity()
    def forward(s, x): return torch.relu(s.b2(s.c2(torch.relu(s.b1(s.c1(x))))) + s.sk(x))

class ResNet(nn.Module):
    def __init__(s, w=40):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(4, w, (5, 3), 1, (2, 1), bias=False), nn.BatchNorm2d(w), nn.ReLU())
        s.body = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1),
                               Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1), Block(4 * w, 6 * w, 2), Block(6 * w, 6 * w, 1),
                               Block(6 * w, 8 * w, 2))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(16 * w, 9))
    def forward(s, x):
        h = s.body(s.stem(x)); return s.head(torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1))

def make(arch):
    if arch == 'res': return ResNet()
    import timm
    name = {'timm_resnet18': 'resnet18', 'timm_effb0': 'efficientnet_b0'}[arch]
    try: return timm.create_model(name, pretrained=True, in_chans=4, num_classes=9)
    except Exception as e: print('pretrained download failed, random init:', e); return timm.create_model(name, pretrained=False, in_chans=4, num_classes=9)

def predict(net, Z, M, G, tta=True):
    net.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for j in range(0, len(Z), 512):
            z = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float()
            g = torch.from_numpy(G[j:j + 512]).to(dev)
            p = torch.softmax(net(canvas(z, m, g)).float(), 1)
            if tta: p = 0.5 * (p + torch.softmax(net(canvas(z.flip(-1), m.flip(-1), g)).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)

def pks_like(Z, M, seed=0):
    """deterministic 'PKS-like' version of held-out GBT data for shift-aware validation"""
    torch.manual_seed(seed); out_z, out_m = [], []
    for j in range(0, len(Z), 512):
        z = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float(); B = len(z)
        z = 0.75 * z + math.sqrt(1 - 0.75 ** 2) * torch.randn_like(z)
        xs = torch.linspace(-1, 1, 128, device=dev) / 0.64
        grid = torch.stack([xs[None, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=dev)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, align_corners=True).reshape(B, 6, 16, 128)
        out = (xs.abs() > 1)[None, None, None, :].expand_as(zz)
        zz = torch.where(out, torch.randn_like(zz), zz); sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        mm = torch.where(out, torch.ones_like(m), Fn.grid_sample(m.reshape(B, 1, 96, 128), grid, mode='nearest', align_corners=True).reshape(B, 6, 16, 128))
        drop = (torch.rand(B, 6, 16, 1, device=dev) < 0.02).float(); mm = mm * (1 - drop); zz = zz * (1 - drop)
        out_z.append(zz.cpu().numpy().astype(np.float16)); out_m.append(mm.cpu().numpy() > 0.5)
    return np.concatenate(out_z), np.concatenate(out_m)

def fit(Z, M, G, y, w_cls, epochs, seed, arch, evals):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make(arch).to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=0.02)
    bs = int(os.environ.get('BS', 128)); n = len(y); steps = epochs * (n // bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(w_cls, device=dev, dtype=torch.float32), label_smoothing=0.05)
    scaler = torch.amp.GradScaler(enabled=dev == 'cuda'); Y = torch.from_numpy(y)
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(n); t0 = time.time(); L = 0
        for j in range(0, n - bs + 1, bs):
            idx = perm[j:j + bs]
            z = torch.from_numpy(Z[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev).float()
            g = torch.from_numpy(G[idx]).to(dev); yb = Y[idx].to(dev)
            if P_ETI + P_TRAP > 0: z, yb = inject(z, m, g, yb, P_ETI, P_TRAP)
            z, m = augment(z, m, True)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(net(canvas(z, m, g)), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); L += loss.item() * bs
        if ep % 5 == 4 or ep == epochs - 1: print(f'   ep{ep} loss {L / n:.3f} {time.time() - t0:.1f}s', flush=True)
    return [predict(net, *e) for e in evals]

if __name__ == '__main__':
    D, EP, SEEDS = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
    arch = sys.argv[4] if len(sys.argv) > 4 else 'res'; PL = 'PL' in sys.argv
    tag = os.environ.get('TAG', arch); SMOKE = int(os.environ.get('SMOKE', 0)); os.makedirs('out', exist_ok=True)
    Z, M, G, meta = load(D, 'train'); Zt, Mt, Gt, tmeta = load(D, 'test')
    y = meta.label.map({l: i for i, l in enumerate(LABELS)}).values
    if SMOKE: Z, M, G, y = Z[:SMOKE], M[:SMOKE], G[:SMOKE], y[:SMOKE]; Zt, Mt, Gt = Zt[:SMOKE], Mt[:SMOKE], Gt[:SMOKE]
    cnt = np.bincount(y, minlength=9); w_cls = (cnt.sum() / (9 * np.maximum(cnt, 1))) ** 0.5; w_cls = w_cls / w_cls.mean()
    print('device', dev, 'arch', arch, 'train', Z.shape, 'class weights', w_cls.round(2), flush=True)
    if PL:
        pl = np.load('out/pl.npz'); sel, ypl = pl['sel'], pl['y']
        print('pseudo-labelled test cadences', sel.sum(), flush=True)
    oof = np.zeros((len(y), 8 + 1)); oof_pks = np.zeros_like(oof); pt = np.zeros((len(Zt), 9))
    for s in range(SEEDS):
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Z, y)):
            Za, Ma, Ga, ya = Z[a], M[a], G[a], y[a]
            if PL: Za, Ma, Ga, ya = np.concatenate([Za, Zt[sel]]), np.concatenate([Ma, Mt[sel]]), np.r_[Ga, Gt[sel]], np.r_[ya, ypl]
            Zp, Mp = pks_like(Z[b], M[b], seed=k)
            pb, pp, pte = fit(Za, Ma, Ga, ya, w_cls, EP, 100 * s + k, arch, [(Z[b], M[b], G[b]), (Zp, Mp, G[b]), (Zt, Mt, Gt)])
            oof[b] += pb / SEEDS; oof_pks[b] += pp / SEEDS; pt += pte / (5 * SEEDS)
            print(f'seed {s} fold {k}: F1 {f1_score(y[b], pb.argmax(1), average="macro"):.4f} | PKS-like F1 {f1_score(y[b], pp.argmax(1), average="macro"):.4f}', flush=True)
        f = SEEDS / (s + 1)
        print(f'== after seed {s}: OOF F1 {f1_score(y, (oof * f).argmax(1), average="macro"):.4f} | PKS-like {f1_score(y, (oof_pks * f).argmax(1), average="macro"):.4f}', flush=True)
        print('per class (PKS-like):', dict(zip(LABELS, f1_score(y, (oof_pks * f).argmax(1), average=None).round(3))), flush=True)
        np.save(f'out/oof_{tag}.npy', oof * f); np.save(f'out/oofpks_{tag}.npy', oof_pks * f); np.save(f'out/test_{tag}.npy', pt * f)
    print('DONE', flush=True)


In [ ]:
import base64, os
os.makedirs('/kaggle/working/out', exist_ok=True)
open('/kaggle/working/out/pl.npz','wb').write(base64.b64decode('UEsDBC0AAAAIAAAAIQDKpT26//////////8HABQAc2VsLm5weQEAEACQJwAAAAAAANMGAAAAAAAAnZc9cmVVDISllFU4M1Q5YFIWQAZFQkBEGcYUAcVQ9hQJsAo2jMc+6v5a1xFv/Mb35xyp1d3Svf732++/+e6Hrj/rr9v3D08/P95+dXP790/vbu9ubn/58Pjx8f73Hz88vn/4dP3r+9+eHp6vP/16/8fD8/nn7758/tx9cXfzz83//HxW1X2+n35a/798Xg+6X9f0OdKt18VV+r7eqbPq9V69xu/SpVbg0opImFtOeq320jbW1nYFBaCXxS14XHFqnnSs79TbxnI2t4GcjV2Aq6QocTaJmNZuRzNHh+g2uYMD0WeRq1DCRrRZokrNfytZO8RE6YAlHiDahJOahsa1kYP0RlzDiftmC5gYSlSJsHFgh0fFkNkBnYoBK1ta71UvNEGIW0tgu7ELvAneLZ6601Sd4LInuHkYpodqbRdeEBJLCBGyw8Dg1u1GiYjd7LAf0EgwWDka+wyWdHmZM5Wi6FrMaiyW3TjDxS1EFlyoLjox5wDYCNdgUxa7WRZHDpfeJV62QRohzUVYjhHDhfOAHkWZEtJySGD7VESxZ5JnUuo6SbXbaDVUEuC2gmv6jQVg2dZKI4fPu/YBjzBRLjHSCugAGCtJxLRAmskfBqTCw46hQhIKwbLgEzaoOwXY9z3y7J4OTQs3oiQOVA8yFurz5RfYn+IGGcZrD/eba8MPJnGktJzNg4q4Zq+9TXaXm7urA4frkR/oS5nKCUMDALAnJHUz8gw3A4UmSy6XEaINBtiPb2B7akvmSRZDAbqyBQKLZz3nqQ0EwS58IWdlcftgkATuLYdx5QYVddjdo96R7N2LQFhzfEJdUNUaDOiE9jbJRphl1Iw2rvAUaKKvuNTxq2ZzCqYUHv8Y/WMs0j/RwwzqokYbuzhxLqUVWlHRSlwcydBKiFOXIPBaMxFLkJaO4H6bsSxbaoNlG17gu8mnPgyRXAtmeMqo2Et3pJ8F2XZ+uwzzhjg5ZVZjmVh7RSxMG8TU2JgRAQHCVlygttDs6sDAYcmxMMVa6uwnBG8lDi3cGG1vw07bWcUbDIPFbgZaEHTI1vLBVDYPGhNifIVQ7gEIYPzoDvROqCuiKO7QqamtS+WUYEj+QLFj06nIkkGb1YUt8H5eyCxQP8Ct/sYPRo1jWkL2S7u6cZ7dw57rWa6Wo5uuRggKCTMgigXMUGhq/5lATETLyR6AycqfSbAaLkTcN3r9DsVcANePtlKP4kbcsPm54B5hr9jB2QiqGvj4vrGyxjEdtFc7lcRMGqYdYNguwEwCY7CwWApJ/xDRTAWCQj0N3mwXK+BmyXSyguVJGRuwL43lljFG7tSNKRRtkW6a4obRdjHlfD5XlhjUoqOIfEncmzsoFaxTEBY2C9GzcOZlR1hjxSr5r3nYu0RLYlu3iWFEjQ7TrrW9kioX+tjDZ83ViAtpk2GjhZ/GehVbL21IISS8m9GGEgRVkJfxGJsldOPO25FdXLAh1Fk5qF2mEE+g5hZTSfl0LezrAGUlWFlo6KtR/AZTcbRvHgymko5ybQbexHJK1NPPNixhczTo1Npp6I08Ibfpcwg+JscCbiSMIGRXZAt5pcIJ4Vx3Fjwkv2YDhNKAhJSBgfWlaaMuz2oI06RYmokmMArv0Fl8JThp7CQrjDv7/QEkiYCISTeIFvoFPtbzRrBdJJ4K2TLHgIgQ2uMxMczNORCEsKtnhvaMDSufbzu0o3Uc8r0S2mudvNEEtOizKnoaSzx3gHFfqJB8MaKKBQhSz3gIXysbTxwphLfXjvJoXnuyA+xyuUB5CAzEkHH8CIqnIhgdsEl/jPgKhEh+FcNTibVYDZl6xFLXpA38rIAcdt0qtBWgicOTUeKipOWrYUhdt9ihnfj4cziM3WQL/+BySFKMNwhoAhlFvWHdDRfO5JNPKWwJO70VAa6BMygonIr55Vu2RxYFmrY3VJx4AzKPZzN34ZDmAy1mO+aZuclCF1aEVE3KRCVKngH3uipthBnGsplAmqh2l1KVHKcUW4E7aFBO9w6QQvPQBnHUSS6iGY1/i1l/h7BnIcJIYv3psUKri2fx42nIiuYCmwT2IWvwFYQJzTVtFouuKW0cfihwwogSLNqnjIYjIrJFSj8xGHJQB4OpQ0qzEnSohzD2M4oDZ+ibs9+UOa1MGOklbOTkpST9rbkcWLWt11kqdmUGlapuSSJpTQHAL39Gi4anp5E6zrOwiBdIg3rU40ayd/inVhPaxeydiFv1BhGlwgZuGEcxO2oEoekYhpT33de03lpkMS9z/Dwhp5n4DjY7Ki+VIPsN3LOYL/8mfRnSTCE70q8diz8MLbOuheHHyOrs7Ruyu95g3NKQ4eXuf1BLAwQtAAAACAAAACEAjT4hiP//////////BQAUAHkubnB5AQAQAKhwAAAAAAAAyAkAAAAAAACd3M+qHdcRxWFZf3yFn0Kzm4BmJmAbjz1LyCSDjIKIZBwwtpGMJ8ZPkReOQ9STBR+/3bmT5vQ93V27dtWqVavOOf/+y9/+/Ne/f/Lklye/Pr599+Gf7x+/evX49b++eHz96vHbH9///P7ND//48f3bd/89/82b7z+8+/38h+/e/PTu99d/+PxPX3z5+o+vX/326v/9++zpk//9Pft4fPnx+OnH44uPx6fz/6fzvk/n/HO8/mSOev/LOX/df6+/ji/mte6753e9D3Offe6ue8+vXQ/4/9rzbF5f91+/7X48n/Prrz2ufeuf633rfz1f638677/u82yO8veuf+93HTcedz/XfsX1+vE6rn173T53/XqdL3tl3+7HxrnsqXhTHu6+y++Vt8rD9Yvs2OPG0fph91k4pX26/v9s/l/7uc/b+5+us+L0Oq59i7u7buG77Jcd+3rz+br+Yc7LD+svxcXGq/Z146vyffF240Bxv//f5+/767h2CMeqnml/N64f5vxpPb4bp7sO7bfut3aKlygfNg73ORtviivFgfJvcV5xq/P7Wn4tPrR26f+ya5/zMO9XfioPCs+rnqouKH/KLtU37Vu9X/uy12+87nrXPvGv62/XK3zWc5X/qqvFG4QTp3X7FOc3vovf6/5VB2Wf4kF8465d4k2FR5U/ek5dJ39tf6P7Ku8VV5vXhQOKu62D6l90f+W/9l37pfipunu9LvxV3lZ/X/VR69S+VV+uuBePU30VLxWfVHzs+8VfKh9Uj6pf3n3YuqI4qrhae7Qvu+67PEhxXHWt+Ir48Wke7ro3/gv/5JfNl43TPV/+qfosPClcrPVtXF9/lx2Ll3f72VO9aO0pfUX+ONUh9Fzt66m+oucpnxWHqzeJNxW+nPZji6eLo4Wnsqv6/8Vv+X33ZXGy4m1x7FQ/kz4k/4rHrb+Ex6UzV/+z/hH/lM5T9z3Vt9bPyks957RvVfxX3dw6UHmt+cY+T/xj87r4c9VV6X2bL4s74r+73utv91V96d5PunXxSvE49ffKQ/m/+n/pSrsPi8+Fw+t/8bfa/+Lj6s+Ec9c6ik/t9bsvyou7/U7x3NLdTnGz8KL4tPih6kvxMs2plH97nfhCrfeufbuuzY/qX2Wf8KzqQOG96q/qjHiO8E3rUT4WTxCuKT5P9UvlbfHxrc+lS4tHy17xHukGwvvFS8VP4ULx3FNdQX204lh4KNxUnRFPUV9cczPFQ9Ur4cSpXqV+v/pG7bviQ+upeq73CW80p9K69HmD6nekh52ub/2tfajPx4jHSedTHhR/EZ4XbyndYvOk+GPlsewVTz7lP4WLqt9VnytfK+7LvuI1pTNvHqvuq05s3a35k/wm3ULzduG4+OzWXekKVS+Vd8X3K++kw1R/WH2Urq+53+aT8H/nKKqjyuPC+1MdsuaAijvFjeJb+6P/S08QX5V/F38Kz1THqr6V/ZXfyrf1k3BR67url4kPVJ1d3qA6UPWr+hfFj/JFeSx/VB+q9Uq3qfl75XXNW4SP0mFKt1be63zFjfJr7Vk7LvtVt7Tf0h+l8wu3te7CUemo0t/0+eji++oHT/v6jQ/po1qPeJ3wRvYrDtZfqtPCpeqDxJ+qn5UOXPVex8V94bPwSPZW/144oP5c8as8VLyuH8Uzqx+TH0v/KP1i5+HiIbtfpeuUznaqu4t3P5n3icfILu2f5kiqq8XXlCfCveoXTvPxlMdo/epjpcdK71E+3NXvpHMqTqs+bByc6l6XvVsvtv8Uvld9Uz9dukDxktIrTz+vtzihOYryt3QE7b/0m9Jfdb10WtUl4bX4p76HVjr+xpf2s+Jq8/O07iz+qw/bvr4+l1/PrXnt5qm+v3caD4o34V/Nh9aOmmsI/0tfOP184cbPPmd5j3jp+mXrSuF66WVVT3Yd2rfaX/EB1U3lq/jX+qP0V/GU0zns3qd4gviF8EO8ueJZuFG6WuVV2VV1u+qMnqM6q/qg+W3pB4pH3UdxXjhZdaT0heJ5xR+U58L74pnSEev+m9elb64f1q67OK+8KX66epH60z1fOmX5p77PJDyp+cJpPSwds/Q25Y3mHOo3hKc1j9t4qXnU7oNwb/lY6fD7HL0Wryt9Sn2N4qDqi/ys/lR4uNdVvRbfKj215kjV91ad3H1Sn6E+ResqHUH93GldWvv0vWjhgfwvflpxWXOJ4pN350p73V3dtHBC9ux6tm/TUX2z8kb7XPNd1YXiZ+U/1dWtD4urxatLB9bnHJVHuq5+P0L9ivat8kO8puqF+mL14+JV0sk3jsUHql8QHui1/FLzNuVh4dUpntb37Cq/xBsKL0u/3PWezidOdUfhr/JD8SneXTi49tbnJisuVQdqfiu/73pXRyz8qrmf6opwQv3V2qs+r+Yl0jFqzrt4qPwRvxLfK5zTnLnqjOaS8s/i5sahfj+oeGPN62uOLr1LcaZ8VNxu/J7Wn5pnV30uv4h/7fmK240j9UGle4kX1OcK9v2FZ9rf4q+n+yP+U3O+035VfYL4gvraU95eOnD1d7v/1cdUfyj/yI7Nv+UrFYea35c9mpcWfiguTvvO3e/FPeX/xpHiWO8rHlK/D3Oan6f8v3Q0fQ931yX/ijfIjorv4lk7xz/Fl9N8P+Xl0sVU16rvKP2z8KX6qvp8uuJg16Prd/13cV66UvWvypO7PKLySnZunld9Pp1niOeU/0uXUv4Jn9W3lT5XOFR4vted/k7Bfh7sLr/Y9VU/r7oofNd5vT79/l+tq/Ci3qc4qt9pUjzXXEm6inhmPfcUN8X3i09WXyAcVj4q74tXLx6p/xSeLD+pPkv4V/1I1U3105tn9fsxhTebT5qH1Vxd/bz8KD1KurLivvqQfU7Nf8VXig/JP6e/kyqc0fxWPGHfr/xTXNQ+qo6uH07r4R7l57t9Y+GH8lE8UbzgLr/WOjf+lQ97fu3eeBAO6PecxD+qrz2dK2h+r3pTnycsPe+U56gfUB2+238oTiqvr+P2J6fzBvlXeaT+reZ0xS83bqXHCNfkZ91H+Ln4Jr1d+a/fJyt/bFzU519LT1D8PMxR9aE+x1Hzv71O9VRz0dITVU8Xx6WnV78qvD5dt3S32rf9Hfqae21cKr7Ef2sedNqXK+92P7Tv6g+LzwiPhAvq51XH63lbr6tP3/Pyj/rs9dd+/qB4k+Jb+CRdUn2r7K/6qjxYf2t+KL25+G1dX3xX9bjmhurX1adtHCnear4lvlF9t+xWvSmckN3FG6VbFQ9U3yi9pvyoPCx+L7yr+ULpPfV9B10n+4Qn6tOLLygOSj8SXkq3ls6lfrf04tO++1THkU5YenbhcdU94WD1F5oLlm70H1BLAQItAy0AAAAIAAAAIQDKpT260wYAAJAnAAAHAAAAAAAAAAAAAACAAQAAAABzZWwubnB5UEsBAi0DLQAAAAgAAAAhAI0+IYjICQAAqHAAAAUAAAAAAAAAAAAAAIABDAcAAHkubnB5UEsFBgAAAAACAAIAaAAAAAsRAAAAAA=='))
import numpy as np; z=np.load('/kaggle/working/out/pl.npz'); print('pseudo-labels', z['sel'].sum())

In [ ]:
import glob, os, subprocess, time
os.chdir('/kaggle/working')
D = os.path.dirname(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)[0]); print('DATA', D)
!pip -q install timm 2>/dev/null
EPOCHS = 30
p0 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 TAG=res_pl P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} {EPOCHS} 1 res PL > out/log_res_pl.txt 2>&1', shell=True)
p1 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 TAG=effb0_pl P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} {EPOCHS} 1 timm_effb0 PL > out/log_effb0_pl.txt 2>&1', shell=True)
print('started')

In [ ]:
while p0.poll() is None or p1.poll() is None:
    for f in ['out/log_res_pl.txt', 'out/log_effb0_pl.txt']:
        lines = [l for l in open(f).read().splitlines() if 'fold' in l or '==' in l or 'Error' in l or 'per class' in l or 'pseudo' in l]
        print(f, '|', *lines[-3:], sep='\n  ')
    print('-' * 60, time.strftime('%H:%M:%S'), flush=True); time.sleep(300)
print(open('out/log_res_pl.txt').read()[-3000:]); print(open('out/log_effb0_pl.txt').read()[-3000:])

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile('r2_outputs_run3.zip', 'w') as z:
    for f in glob.glob('out/*.npy') + glob.glob('out/*.txt'): z.write(f)
FileLink('r2_outputs_run3.zip')